# 대만 대표기업 월별 매출 — 데이터 확인 & 시각화

`revenue.db`(SQLite)에 저장된 실적/예측 데이터를 확인하고 시각화하는 노트북입니다.

사전 준비: 프로젝트 루트에서 아래를 먼저 실행해 DB에 데이터가 있어야 합니다.
```bash
python main.py collect --start 2019-01   # 과거 이력 수집
python main.py forecast --horizon 6      # 예측
```


In [1]:
# 경로 설정: DATA 폴더를 자동 탐색해 사용자 예측 모듈을 쓸 수 있게 함
from config import setup_universal_paths
paths = setup_universal_paths()   # DATA 폴더가 없어도 오류 없이 내장 엔진 사용

ModuleNotFoundError: No module named 'config'

In [ ]:
# 설정 및 DB 연결
import pandas as pd
import matplotlib.pyplot as plt

from config import COMPANIES
from db import get_conn, latest_basis

pd.set_option("display.float_format", "{:,.1f}".format)
conn = get_conn()
COMPANIES

## 1. 데이터 현황 확인

In [ ]:
# 테이블별 건수와 기업별 수집 범위
summary = pd.read_sql("""
SELECT company_id,
       MAX(company_name)                        AS name,
       COUNT(*)                                 AS n_months,
       MIN(year || '-' || printf('%02d', month)) AS first_month,
       MAX(year || '-' || printf('%02d', month)) AS last_month
FROM revenue GROUP BY company_id ORDER BY company_id
""", conn)
print("revenue rows :", conn.execute("SELECT COUNT(*) FROM revenue").fetchone()[0])
print("forecast rows:", conn.execute("SELECT COUNT(*) FROM forecast").fetchone()[0])
summary

In [2]:
# 최근 6개월 실적 미리보기 (전 기업)
recent = pd.read_sql("""
SELECT company_id, company_name, year, month,
       revenue/1e6 AS revenue_bil, mom_pct, yoy_pct, source
FROM revenue
ORDER BY year DESC, month DESC, company_id
LIMIT 30
""", conn)
recent

NameError: name 'pd' is not defined

In [ ]:
# 예측 이력 확인 (basis = 예측에 사용한 마지막 실적 연월)
fc = pd.read_sql("""
SELECT company_id,
       basis_year  || '-' || printf('%02d', basis_month)  AS basis,
       target_year || '-' || printf('%02d', target_month) AS target,
       predicted/1e6 AS pred_bil,
       lower_95/1e6  AS lo_bil,
       upper_95/1e6  AS hi_bil,
       model
FROM forecast ORDER BY company_id, basis, target
""", conn)
fc.head(15)

## 2. 기업별 시계열 조회

`CID`를 바꿔 원하는 기업을 확인하세요.

In [ ]:
CID = "2330"        # ← 종목코드 변경
MODEL = "ensemble"  # ← 예측 모델 선택: sarima / theta / ets / ensemble

def actual_df(cid):
    df = pd.read_sql(
        "SELECT year, month, revenue, mom_pct, yoy_pct FROM revenue "
        "WHERE company_id = ? ORDER BY year, month", conn, params=(cid,))
    df["date"] = pd.to_datetime(dict(year=df.year, month=df.month, day=1))
    df["revenue_bil"] = df["revenue"] / 1e6   # NTD 천 → 십억
    return df

def forecast_df(cid, basis=None, model=None):
    q = ("SELECT basis_year, basis_month, target_year, target_month, "
         "predicted, lower_95, upper_95, model FROM forecast WHERE company_id = ?")
    params = [cid]
    if basis:
        q += " AND basis_year = ? AND basis_month = ?"
        params += list(basis)
    if model:
        q += " AND model = ?"
        params.append(model)
    df = pd.read_sql(q + " ORDER BY target_year, target_month", conn, params=params)
    if df.empty:
        return df
    df["date"] = pd.to_datetime(dict(year=df.target_year, month=df.target_month, day=1))
    for c in ("predicted", "lower_95", "upper_95"):
        df[c + "_bil"] = df[c] / 1e6
    return df

a = actual_df(CID)
a.tail(12)

## 3. 시각화

### 3-1. 실적 + 최신 예측 (95% 신뢰구간)

In [ ]:
basis = latest_basis(conn, CID)
f = forecast_df(CID, basis=basis, model=MODEL)
name = COMPANIES.get(CID, CID)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(a["date"], a["revenue_bil"], label="Actual", color="#1f77b4", lw=1.6)
if not f.empty:
    last = a.iloc[-1]
    ax.plot([last["date"], *f["date"]], [last["revenue_bil"], *f["predicted_bil"]],
            "--o", color="#d62728", ms=4, lw=1.6, label=f"Forecast ({MODEL})")
    ax.fill_between(f["date"], f["lower_95_bil"], f["upper_95_bil"],
                    color="#d62728", alpha=0.15, label="95% CI")
ax.set_title(f"{name} ({CID}) Monthly Revenue & {MODEL.upper()} Forecast")
ax.set_ylabel("Revenue (NTD billion)")
ax.grid(alpha=0.3); ax.legend(); fig.autofmt_xdate()
plt.show()

### 3-2. 과거 예측 이력 vs 실적 (매월 예측 추적)

In [ ]:
all_f = forecast_df(CID, model=MODEL)
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(a["date"], a["revenue_bil"], label="Actual", color="#1f77b4", lw=2, zorder=5)
if not all_f.empty:
    bases = sorted(set(zip(all_f.basis_year, all_f.basis_month)))
    cmap = plt.get_cmap("autumn")
    for i, (by, bm) in enumerate(bases):
        g = all_f[(all_f.basis_year == by) & (all_f.basis_month == bm)]
        ax.plot(g["date"], g["predicted_bil"], "--.", ms=6, lw=1.1, alpha=0.85,
                color=cmap(i / max(len(bases) - 1, 1) * 0.8),
                label=f"Forecast @ {by}-{bm:02d}")
ax.set_title(f"{name} ({CID}) {MODEL.upper()} Forecast History vs Actual")
ax.set_ylabel("Revenue (NTD billion)")
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2); fig.autofmt_xdate()
plt.show()

### 3-2b. 4개 모델 최신 예측 비교

In [ ]:
colors = {"sarima": "#d62728", "theta": "#2ca02c", "ets": "#9467bd", "ensemble": "#ff7f0e"}
fall = forecast_df(CID, basis=basis)  # 최신 basis의 전체 모델
fig, ax = plt.subplots(figsize=(11, 5))
tail = a.tail(24)
ax.plot(tail["date"], tail["revenue_bil"], label="Actual", color="#1f77b4", lw=2)
last = a.iloc[-1]
if not fall.empty:
    fall["date"] = pd.to_datetime(dict(year=fall.target_year, month=fall.target_month, day=1))
    fall["pred_bil"] = fall["predicted"] / 1e6
    for mname, c in colors.items():
        g = fall[fall.model == mname].sort_values("date")
        if g.empty:
            continue
        lw = 2.2 if mname == "ensemble" else 1.2
        ax.plot([last["date"], *g["date"]], [last["revenue_bil"], *g["pred_bil"]],
                "--o", ms=4, lw=lw, color=c, label=mname)
ax.set_title(f"{name} ({CID}) Forecast Model Comparison")
ax.set_ylabel("Revenue (NTD billion)")
ax.grid(alpha=0.3); ax.legend(); fig.autofmt_xdate()
plt.show()

### 3-3. 전년 동월 대비 증감률(YoY)

In [ ]:
yoy = a.dropna(subset=["yoy_pct"]) if a["yoy_pct"].notna().any() else None
if yoy is None or yoy.empty:
    # DB에 yoy가 없으면 직접 계산
    tmp = a.set_index("date")["revenue"].astype(float)
    yoy_series = (tmp / tmp.shift(12) - 1) * 100
    yoy = yoy_series.dropna().rename("yoy_pct").reset_index()

fig, ax = plt.subplots(figsize=(11, 4))
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in yoy["yoy_pct"]]
ax.bar(yoy["date"], yoy["yoy_pct"], width=20, color=colors)
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"{name} ({CID}) YoY Growth (%)")
ax.set_ylabel("%"); ax.grid(alpha=0.3, axis="y"); fig.autofmt_xdate()
plt.show()

### 3-4. 전체 기업 비교 (지수화: 시작월 = 100)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for cid, cname in COMPANIES.items():
    d = actual_df(cid)
    if d.empty:
        continue
    idx = d["revenue"].astype(float) / float(d["revenue"].iloc[0]) * 100
    ax.plot(d["date"], idx, lw=1.5, label=f"{cname} ({cid})")
ax.set_title("Revenue Index Comparison (first month = 100)")
ax.set_ylabel("Index"); ax.grid(alpha=0.3); ax.legend(); fig.autofmt_xdate()
plt.show()

## 4. (선택) 노트북에서 바로 예측 실행

horizon을 바꿔가며 예측을 실행하고 결과를 즉시 확인할 수 있습니다. 결과는 DB에도 저장됩니다.

In [ ]:
HORIZON = 6   # ← 예측 개월 수 조정

from forecaster import forecast_company
result = forecast_company(conn, CID, horizon=HORIZON)  # 4개 모델 모두 실행/저장
pd.DataFrame(result) if result else "관측치 부족 또는 실패" 

In [ ]:
conn.close()